<Strong><span style="font-size:40px;color:lightgreen">Crypto index fund model</strong>

<span style="color:orange;font-size:30px">Imports

In [14]:
print("Starting Imports...")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
import datetime
import os
import yfinance as yf
import logging
print("Imports completed.")

Starting Imports...
Imports completed.


<span style="color:orange;font-size:30px">Environment

In [ ]:

class CryptoIndexFundTradingEnv(gym.Env):
    def __init__(self, data, feature_cols=None, transaction_cost=0.001, normalize_features=True):
        super().__init__()

        self.full_data = data.copy()
        self.tickers = sorted(list(self.full_data['Ticker'].unique()))  # consistent order
        self.feature_cols = feature_cols or [
            col for col in self.full_data.select_dtypes(include=np.number).columns if col != 'Close'
        ]
        self.transaction_cost = transaction_cost
        self.normalize_features = normalize_features

        # --- Pre-process into arrays ---
        self.data_dict = {}
        min_len = float('inf')
        for ticker in self.tickers:
            ticker_df = self.full_data[self.full_data['Ticker'] == ticker]
            min_len = min(min_len, len(ticker_df))

            features_df = ticker_df[self.feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
            if normalize_features:
                # Min-max scaling per feature column
                min_vals = features_df.min()
                max_vals = features_df.max()
                features_df = (features_df - min_vals) / (max_vals - min_vals + 1e-9)

            self.data_dict[ticker] = {
                'features': features_df.to_numpy(dtype=np.float32),
                'close': ticker_df['Close'].to_numpy(dtype=np.float32)
            }

        # Trim all to same length
        self.max_steps = min_len
        for t in self.tickers:
            self.data_dict[t]['features'] = self.data_dict[t]['features'][:self.max_steps]
            self.data_dict[t]['close'] = self.data_dict[t]['close'][:self.max_steps]

        self.close_prices_matrix = np.array(
            [self.data_dict[t]['close'] for t in self.tickers],
            dtype=np.float32
        )

        # --- Spaces ---
        # Action = allocation weights for each asset (continuous between 0-1, softmax inside step)
        self.action_space = spaces.Box(low=0.0, high=1.0, shape=(len(self.tickers),), dtype=np.float32)

        obs_size = len(self.feature_cols) * len(self.tickers) + len(self.tickers) + 1
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_size,), dtype=np.float32)

        self.initial_cash = 10000.0
        self.reset()

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)

        self.current_step = 0
        self.cash = 0.0
        self.weights = np.ones(len(self.tickers)) / len(self.tickers)
        self.holdings = np.zeros(len(self.tickers), dtype=np.float32)
        self.portfolio_value_history = [self._get_portfolio_value(self.current_step)]
        self.action_history = []

        # Initial purchase
        initial_prices = self.close_prices_matrix[:, self.current_step]
        allocated_cash = self.initial_cash * self.weights
        self.holdings = allocated_cash / (initial_prices + 1e-9)

        return self._get_state(), {}

    def _get_state(self):
        features = np.concatenate([self.data_dict[t]['features'][self.current_step] for t in self.tickers])
        state = np.concatenate([features, self.holdings, [self.cash]], axis=0).astype(np.float32)
        return state

    def _get_portfolio_value(self, step):
        prices = self.close_prices_matrix[:, step]
        return self.cash + np.dot(self.holdings, prices)

    def step(self, action):
        terminated = False
        truncated = False

        # Ensure valid allocation weights
        action = np.clip(action, 0, 1)
        if action.sum() > 0:
            action /= action.sum()  # softmax-like normalisation
        else:
            action = np.ones_like(action) / len(action)

        prev_value = self._get_portfolio_value(self.current_step)
        self.action_history.append(action)

        # Move to next step
        self.current_step += 1
        if self.current_step >= self.max_steps - 1:
            truncated = True

        # Rebalance portfolio
        current_prices = self.close_prices_matrix[:, self.current_step]
        target_value_per_asset = prev_value * action
        target_holdings = target_value_per_asset / (current_prices + 1e-9)

        # Transaction cost on change in holdings
        trade_amount = np.abs(target_holdings - self.holdings) * current_prices
        cost = trade_amount.sum() * self.transaction_cost

        self.holdings = target_holdings
        self.cash = prev_value - np.dot(self.holdings, current_prices) - cost
        self.cash = max(self.cash, 0.0)  # avoid negative cash from rounding errors

        current_value = self._get_portfolio_value(self.current_step)
        self.portfolio_value_history.append(current_value)

        # Reward = % change in portfolio value
        reward = (current_value - prev_value) / (prev_value + 1e-9)
        reward = float(np.clip(reward, -1, 1))  # clip for stability

        info = {
            "portfolio_value": current_value,
            "holdings": self.holdings.copy(),
            "weights": action.copy(),
            "cash": self.cash
        }

        return self._get_state(), reward, terminated, truncated, info

<span style="color:green;font-size:25px">Data formatting from yfinance

In [33]:
def rolling_statistics(df, target_col, group_col, window):
    """
    df: Pandas DataFrame
    target_col: string, name of the column to calculate rolling stats on
    group_col: string, name of the grouping column
    window: int, window size
    """
    rolling_max = df.groupby(group_col)[target_col].rolling(window=window).max().reset_index(level=0, drop=True)
    rolling_min = df.groupby(group_col)[target_col].rolling(window=window).min().reset_index(level=0, drop=True)
    rolling_std = df.groupby(group_col)[target_col].rolling(window=window).std().reset_index(level=0, drop=True)
    rolling_avg = df.groupby(group_col)[target_col].rolling(window=window).mean().reset_index(level=0, drop=True)
    return rolling_max, rolling_min, rolling_std, rolling_avg

def rsi (df, target_col, group_col, window):
    """
    calcualtes the RSI score (Relative Strength Index) for a given data series
    :paramaters
        data: pandas Series
        Window: int, length of rolling window
    """
    delta = df.groupby(group_col)[target_col].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.groupby(df[group_col]).rolling(window=window).mean().reset_index(level=0, drop=True)
    avg_loss = loss.groupby(df[group_col]).rolling(window=window).mean().reset_index(level=0, drop=True)
    rs = avg_gain / (avg_loss + 1e-10)
    rsi = 100 - (100 / (1 + rs))
    return rsi

def macd(df, target_col, short_window, long_window, signal_window, group_col):
    """
    Moving Average Convergence Divergence (MACD) calculation.
    Measures trend strength and direction.
    Parameters:
        df: Pandas DataFrame
        target_col: str, column name to compute MACD on
        short_window: int, short-term EMA window (default 12)
        long_window: int, long-term EMA window (default 26)
        signal_window: int, signal line EMA window (default 9)
        group_col: str, column name to group by (e.g., 'Ticker')
    Returns:
        macd_line: Pandas Series aligned with df.index
        macd_signal: Pandas Series aligned with df.index
    """
    ema_short = df.groupby(group_col)[target_col].transform(lambda x: x.ewm(span=short_window, adjust=False).mean())
    ema_long = df.groupby(group_col)[target_col].transform(lambda x: x.ewm(span=long_window, adjust=False).mean())
    macd_line = ema_short - ema_long

    # Calculate signal line by applying ewm *per group* but keep original index using transform
    macd_signal = macd_line.groupby(df[group_col]).transform(lambda x: x.ewm(span=signal_window, adjust=False).mean())
    return macd_line.astype(np.float32), macd_signal.astype(np.float32)



data colection and feature enginnering 

In [35]:
tickers = ['BTC-USD', 'ETH-USD', 'XRP-USD', 'LTC-USD', 'BCH-USD']

yfinance_data = pd.DataFrame()

print(f"Downloading data from Yahoo Finance...")
try:
    ticker_data = yf.download(tickers, start="2018-01-01", end=datetime.datetime.now().strftime("%Y-%m-%d"))
    print(f"Data download completed.")
except Exception as e:
    logging.error(f"Error downloading data.")
    raise

print(f"unmprocessed ticker data")
print(ticker_data.head())

ticker_data = ticker_data.stack(level=1).reset_index()
# Rename columns to something clearer
ticker_data.columns = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
# Optional: sort for neatness
ticker_data = ticker_data.sort_values(['Date', 'Ticker'], ascending=[True,True]).reset_index(drop=True)
print("corrected ticker cols and index")
print(ticker_data.head())

feature_cols_to_engineer = ['Close']
# Add engineered features
for col in feature_cols_to_engineer:
    ticker_data[f'{col}_rolling_max'], ticker_data[f'{col}_rolling_min'], ticker_data[f'{col}_rolling_std'], ticker_data[f'{col}_rolling_avg'] = rolling_statistics(ticker_data, col, 'Ticker', window=14)
    ticker_data[f'{col}_rsi'] = rsi(ticker_data, col, 'Ticker', window=14)
    macd_line, macd_signal = macd(ticker_data, col, 12, 26, 9, 'Ticker')
    ticker_data[f'{col}_macd_line'] = macd_line
    ticker_data[f'{col}_macd_signal'] = macd_signal
print("fully processed ticker data with feature engineering")
print(ticker_data.head())

[*********************100%***********************]  5 of 5 completed

Data download completed.
unmprocessed ticker data
Price             Close                                                 \
Ticker          BCH-USD       BTC-USD     ETH-USD     LTC-USD  XRP-USD   
Date                                                                     
2018-01-01  2432.540039  13657.200195  772.640991  229.033005  2.39103   
2018-01-02  2711.000000  14982.099609  884.443970  255.684006  2.48090   
2018-01-03  2608.689941  15201.000000  962.719971  245.367996  3.10537   
2018-01-04  2430.179932  15599.200195  980.921997  241.369995  3.19663   
2018-01-05  2584.479980  17429.500000  997.719971  249.270996  3.04871   

Price              High                                                  ...  \
Ticker          BCH-USD       BTC-USD      ETH-USD     LTC-USD  XRP-USD  ...   
Date                                                                     ...   
2018-01-01  2534.860107  14112.200195   782.530029  236.634003  2.39103  ...   
2018-01-02  2867.139893  15444.599609


/var/folders/f2/_cjvdffn2x91f953fzbj08dc0000gn/T/ipykernel_40774/447630041.py:16: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  ticker_data = ticker_data.stack(level=1).reset_index()


- training loop bulid <br>
- training chart <br>
- Logging of model activities <br>
- Model Storage

- Testing loop build<br>
- Testing performance - portfolio growth<br>
- actions <br>
- Risk appitite analysis <br>